# Construction du RAG final — Guidelines + Kanakmi + Shifaa

**Ce que fait ce notebook :**
1. Parcourt `Guidelines/<Pathologie>/*.md` et détecte langue + pathologie automatiquement depuis l'arborescence/nom de fichier
2. Nettoie le bruit récurrent (copyright NICE, numéros de page)
3. Chunk chaque guideline par section (titres Markdown)
4. Charge les CSV déjà préparés de Kanakmi et Shifaa (schéma déjà unifié : `id, text, lang, role_cible, source, disorder`)
5. Fusionne tout en une seule table
6. Génère les embeddings (bge-m3, multilingue)
7. Sauvegarde dans une base ChromaDB persistante sur Drive

In [ ]:
!pip install -q sentence-transformers chromadb pandas

import os
import re
import json
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE = '/content/drive/MyDrive/Master_Thesis_RAG_Indexes/Master_Thesis_RAG_Indexes'
GUIDELINES_DIR = f"{BASE_DRIVE}/Guidelines"
KANAKMI_CSV = f"{BASE_DRIVE}/Kanakmi/mental_disorders_prepared_rag.csv"
SHIFAA_CSV = f"{BASE_DRIVE}/Shifaa/shifaa_prepared_rag.csv"
VECTOR_DB_DIR = f"{BASE_DRIVE}/Vector_Database/chroma_db_v1"
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

print("📁 Guidelines:", GUIDELINES_DIR)
print("📁 Vector DB :", VECTOR_DB_DIR)

Mounted at /content/drive
📁 Guidelines: /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Master_Thesis_RAG_Indexes/Guidelines
📁 Vector DB : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Master_Thesis_RAG_Indexes/Vector_Database/chroma_db_v1


## 1. Détection automatique langue / pathologie / rôle depuis l'arborescence

In [ ]:
# Dossier -> nom de pathologie normalisé
FOLDER_TO_DISORDER = {
    "PTSD": "PTSD",
    "Bipolar": "Bipolar",
    "Schizophrenia": "Schizophrenia",
    "Depression": "Depression",
    "BPD": "BPD",
    "WHO": "Transversal"  # mhGAP couvre les 5 pathologies
}

def detect_lang(filename):
    """Détecte la langue depuis le nom de fichier (suffixe _FR, _AR, _EN ou défaut EN)."""
    name = filename.lower()
    if re.search(r'_fr\b|_french|french', name):
        return 'fr'
    if re.search(r'_ar\b|_arabic|arabic', name):
        return 'ar'
    return 'en'

def detect_role(folder_name):
    """Toutes les guidelines (y compris WHO/mhGAP) sont accessibles aux deux rôles :
    R1 les utilise comme socle normatif, R2 comme source de connaissance validée
    (contrairement à Kanakmi qui reste réservé à R2 comme illustration de vécu)."""
    return True, True  # role_clinical, role_education

print("✅ Fonctions de détection prêtes")

✅ Fonctions de détection prêtes


## 2. Nettoyage du bruit récurrent (copyright, pagination)

In [ ]:
def clean_guideline_text(text):
    # Copyright NICE répété
    text = re.sub(r"©\s*NICE\s*\d{4}\..*?rights\)\.", "", text, flags=re.DOTALL)
    # Numéros de page
    text = re.sub(r"Page \d+ of \d*\s*", "", text)
    # Artefacts de commentaire HTML issus de la conversion PDF->MD
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    # Balises <u> résiduelles
    text = re.sub(r"</?u>", "", text)
    # Lignes vides multiples
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 3. Chunking par section Markdown (## ou ###), pas par nombre de tokens brut

In [ ]:
def chunk_by_section(text, max_chars=2500):
    """Découpe par titres Markdown (## / ###). Si une section dépasse max_chars,
    la sous-découpe par paragraphe pour rester dans une taille raisonnable pour l'embedding."""
    # Découpe sur les titres de niveau 2 ou 3
    sections = re.split(r'(?=^#{2,3}\s)', text, flags=re.MULTILINE)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        title_match = re.match(r'^#{2,3}\s*(.+)', section)
        title = title_match.group(1).strip() if title_match else "Introduction"

        if len(section) <= max_chars:
            chunks.append((title, section))
        else:
            # sous-découpe par paragraphe si la section est trop longue
            paragraphs = section.split('\n\n')
            buffer = ""
            for p in paragraphs:
                if len(buffer) + len(p) > max_chars and buffer:
                    chunks.append((title, buffer.strip()))
                    buffer = p
                else:
                    buffer += "\n\n" + p
            if buffer.strip():
                chunks.append((title, buffer.strip()))
    return chunks

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/'))

['Colab Notebooks', 'Master_Thesis_RAG_Indexes']


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/Master_Thesis_RAG_Indexes'))

['Master_Thesis_RAG_Indexes', 'Vector_Database']


## 4. Parcours de l'arborescence Guidelines/ et construction des chunks

In [ ]:
guideline_records = []
chunk_counter = 0

for disorder_folder in Path(GUIDELINES_DIR).iterdir():
    if not disorder_folder.is_dir():
        continue
    disorder_name = FOLDER_TO_DISORDER.get(disorder_folder.name, disorder_folder.name)
    role_clinical, role_education = detect_role(disorder_folder.name)

    for md_file in disorder_folder.glob("*.md"):
        lang = detect_lang(md_file.name)
        raw_text = md_file.read_text(encoding='utf-8')
        clean = clean_guideline_text(raw_text)
        sections = chunk_by_section(clean)

        for title, section_text in sections:
            chunk_counter += 1
            guideline_records.append({
                'id': f"guideline_{chunk_counter}",
                'text': section_text,
                'lang': lang,
                'role_clinical': role_clinical,
                'role_education': role_education,
                'content_type': 'clinical_guideline',
                'source': md_file.stem,
                'disorder': disorder_name,
                'section_title': title
            })

        print(f"✅ {md_file.name} -> {len(sections)} chunks ({disorder_name}, {lang})")

df_guidelines = pd.DataFrame(guideline_records)
print(f"\n📦 Total chunks guidelines : {len(df_guidelines)}")
print(df_guidelines['disorder'].value_counts())

✅ NICE_CG185_Bipolar_2024_EN.md -> 46 chunks (Bipolar, en)
✅ CANMAT_ISBD_Bipolar_2025_EN.md -> 135 chunks (Bipolar, en)
✅ NICE_CG78_BPD_2009_EN.md -> 60 chunks (BPD, en)
✅ NICE_NG222_Depression_2022_FR.md -> 245 chunks (Depression, fr)
✅ NICE_NG222_Depression_2022_EN.md -> 98 chunks (Depression, en)
✅ VA_DoD_PTSD_2023_EN.md -> 253 chunks (PTSD, en)
✅ APA_PTSD_2025_EN.md -> 291 chunks (PTSD, en)
✅ APA_Schizophrenia_2020_EN.md -> 522 chunks (Schizophrenia, en)
✅ mhGAP_AR.md -> 55 chunks (Transversal, ar)
✅ mhGAP_FR.md -> 138 chunks (Transversal, fr)
✅ mhGAP_EN.md -> 120 chunks (Transversal, en)

📦 Total chunks guidelines : 1963
disorder
PTSD             544
Schizophrenia    522
Depression       343
Transversal      313
Bipolar          181
BPD               60
Name: count, dtype: int64


## 5. Chargement de Kanakmi et Shifaa (déjà préparés, même schéma)

In [ ]:
df_kanakmi = pd.read_csv(KANAKMI_CSV)
df_shifaa = pd.read_csv(SHIFAA_CSV)

print(f"Kanakmi : {len(df_kanakmi)} lignes, colonnes : {list(df_kanakmi.columns)}")
print(f"Shifaa  : {len(df_shifaa)} lignes, colonnes : {list(df_shifaa.columns)}")

# section_title n'existe pas pour ces deux sources -> on l'ajoute vide pour l'harmonisation du schéma final
df_kanakmi['section_title'] = None
df_shifaa['section_title'] = None

Kanakmi : 15000 lignes, colonnes : ['id', 'text', 'lang', 'role_clinical', 'role_education', 'content_type', 'source', 'disorder']
Shifaa  : 34912 lignes, colonnes : ['id', 'text', 'lang', 'role_clinical', 'role_education', 'content_type', 'source', 'disorder']


## 6. Fusion en une table unique

In [ ]:
COMMON_COLUMNS = ['id', 'text', 'lang', 'role_clinical', 'role_education', 'content_type', 'source', 'disorder', 'section_title']

df_master = pd.concat([
    df_guidelines[COMMON_COLUMNS],
    df_kanakmi[COMMON_COLUMNS],
    df_shifaa[COMMON_COLUMNS]
], ignore_index=True)

# Sécurité : pas de texte vide, pas de doublon exact
df_master = df_master[df_master['text'].str.strip().str.len() > 0]
df_master = df_master.drop_duplicates(subset=['text'])

print(f"📦 Table finale : {len(df_master)} chunks")
print("\nRépartition role_clinical:")
print(df_master['role_clinical'].value_counts())
print("\nRépartition role_education:")
print(df_master['role_education'].value_counts())
print("\nRépartition content_type:")
print(df_master['content_type'].value_counts())
print("\nRépartition par langue:")
print(df_master['lang'].value_counts())
print("\nRépartition par source:")
print(df_master['source'].value_counts())

📦 Table finale : 51854 chunks

Répartition role_clinical:
role_clinical
True     36854
False    15000
Name: count, dtype: int64

Répartition role_education:
role_education
True    51854
Name: count, dtype: int64

Répartition content_type:
content_type
clinical_dialogue     34912
patient_testimony     15000
clinical_guideline     1942
Name: count, dtype: int64

Répartition par langue:
lang
ar    34967
en    16511
fr      376
Name: count, dtype: int64

Répartition par source:
source
Shifaa_Arabic_Mental_Health_Consultations    34912
Kanakmi/mental-disorders                     15000
APA_Schizophrenia_2020_EN                      522
APA_PTSD_2025_EN                               291
VA_DoD_PTSD_2023_EN                            242
NICE_NG222_Depression_2022_FR                  238
mhGAP_FR                                       138
CANMAT_ISBD_Bipolar_2025_EN                    135
mhGAP_EN                                       120
NICE_NG222_Depression_2022_EN                   96
NICE

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun GPU")

True
Tesla T4


## 7. Sauvegarde de la table fusionnée (avant embedding, pour traçabilité)

In [ ]:
MERGED_DIR = f"{BASE_DRIVE}/Processed_Documents"
os.makedirs(MERGED_DIR, exist_ok=True)
merged_path = f"{MERGED_DIR}/master_chunks_v1.csv"
df_master.to_csv(merged_path, index=False, encoding='utf-8')
print(f"✅ Table fusionnée sauvegardée : {merged_path}")

✅ Table fusionnée sauvegardée : /content/drive/MyDrive/Master_Thesis_RAG_Indexes/Master_Thesis_RAG_Indexes/Processed_Documents/master_chunks_v1.csv


## 8. Génération des embeddings (bge-m3, multilingue AR/EN/FR)

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer("BAAI/bge-m3", device='cuda')
model = model.half() 

texts = df_master['text'].tolist()
print(f"🔄 Génération de {len(texts)} embeddings...")
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
)
print("✅ Embeddings générés")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🔄 Génération de 51854 embeddings...


Batches:   0%|          | 0/811 [00:00<?, ?it/s]

✅ Embeddings générés


## 9. Stockage dans ChromaDB 

In [ ]:
import chromadb

client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_or_create_collection("psych_kb_v1")

# ChromaDB n'accepte pas les valeurs None dans les métadonnées -> on les remplace
df_master['section_title'] = df_master['section_title'].fillna("")

BATCH_SIZE = 500
for i in range(0, len(df_master), BATCH_SIZE):
    batch = df_master.iloc[i:i+BATCH_SIZE]
    batch_embeddings = embeddings[i:i+BATCH_SIZE]
    collection.add(
        ids=batch['id'].tolist(),
        embeddings=[e.tolist() for e in batch_embeddings],
        documents=batch['text'].tolist(),
        metadatas=batch[['lang', 'role_clinical', 'role_education', 'content_type', 'source', 'disorder', 'section_title']].to_dict('records')
    )
    print(f"  → {min(i+BATCH_SIZE, len(df_master))}/{len(df_master)} chunks indexés")

print(f"\n✅ Base vectorielle prête : {collection.count()} documents indexés")
print(f"📁 Sauvegardée dans : {VECTOR_DB_DIR}")

  → 500/51854 chunks indexés
  → 1000/51854 chunks indexés
  → 1500/51854 chunks indexés
  → 2000/51854 chunks indexés
  → 2500/51854 chunks indexés
  → 3000/51854 chunks indexés
  → 3500/51854 chunks indexés
  → 4000/51854 chunks indexés
  → 4500/51854 chunks indexés
  → 5000/51854 chunks indexés
  → 5500/51854 chunks indexés
  → 6000/51854 chunks indexés
  → 6500/51854 chunks indexés
  → 7000/51854 chunks indexés
  → 7500/51854 chunks indexés
  → 8000/51854 chunks indexés
  → 8500/51854 chunks indexés
  → 9000/51854 chunks indexés
  → 9500/51854 chunks indexés
  → 10000/51854 chunks indexés
  → 10500/51854 chunks indexés
  → 11000/51854 chunks indexés
  → 11500/51854 chunks indexés
  → 12000/51854 chunks indexés
  → 12500/51854 chunks indexés
  → 13000/51854 chunks indexés
  → 13500/51854 chunks indexés
  → 14000/51854 chunks indexés
  → 14500/51854 chunks indexés
  → 15000/51854 chunks indexés
  → 15500/51854 chunks indexés
  → 16000/51854 chunks indexés
  → 16500/51854 chunks index

## 10. Test rapide de récupération

In [ ]:
def test_query(query, n_results=3, agent=None, content_type_filter=None):
    """agent: 'R1' ou 'R2'. content_type_filter: ex. 'clinical_guideline' pour forcer R2
    à ignorer les témoignages Kanakmi sauf demande explicite de vécu patient."""
    conditions = []
    if agent == "R1":
        conditions.append({"role_clinical": True})
    elif agent == "R2":
        conditions.append({"role_education": True})
    if content_type_filter:
        conditions.append({"content_type": content_type_filter})

    if len(conditions) == 0:
        where_clause = None
    elif len(conditions) == 1:
        where_clause = conditions[0]
    else:
        where_clause = {"$and": conditions}
    query_embedding = model.encode([query], normalize_embeddings=True)[0].tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where_clause
    )
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        print(f"[{meta['source']} | {meta['disorder']} | {meta['lang']}]")
        print(doc[:300])
        print("---")

# Exemples de test
# R1 : accès direct aux guidelines/Shifaa, jamais Kanakmi
test_query("treatment options for acute mania", agent="R1")

# R2 : par défaut connaissance validée uniquement 
test_query("what is bipolar disorder", agent="R2", content_type_filter="clinical_guideline")

# R2 : uniquement si l'utilisateur demande explicitement un vécu patient
test_query("what does it feel like to live with bipolar disorder", agent="R2", content_type_filter="patient_testimony")

[CANMAT_ISBD_Bipolar_2025_EN | Bipolar | en]
### _Management of psychomotor excitement or agitation_ 

Agitation and psychomotor excitement are common, important, and often immediate foci of attention, when managing acute mania. Verbal de-escalation techniques should be tried first, followed by oral agents, and then parenteral agents. Since ag
---
[CANMAT_ISBD_Bipolar_2025_EN | Bipolar | en]
### **Treatment of acute mania** 

The goals of pharmacological management in acute mania are to rapidly and safely alleviate symptoms, minimize risk of harm to self or others, and restore psychosocial functioning.<sup>[18]</sup> Rapid control of symptoms may strengthen the therapeutic alliance. If 
---
[CANMAT_ISBD_Bipolar_2025_EN | Bipolar | en]
### **Indicative Indian data** 

An Indian Psychiatric Society survey of prescribing practices in the treatment of BD<sup>[53]</sup> found that most respondents (65%) preferred a combination of mood stabilizer (lithium, valproate, and carbamazepine/oxcarb

In [ ]:
import chromadb
client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
collection = client.get_collection("psych_kb_v1")
print(f"Documents indexés : {collection.count()}")

Documents indexés : 51854
